# Supervised Learning with Trees and Forests



PD Dr. Sigve Haug, 2026.

Based on notebooks by Mykhailo Vladymyrov & Aris Marcolongo. https://github.com/neworldemancer/DSF5

This work is licensed under <a href="https://creativecommons.org/share-your-work/public-domain/cc0/">CC0</a>.



## Learning outcomes

- Know pros and cons of trees and forests
- Know how tree models work
- Know how forests work
- Know that feature importance is
- Can train, use and assess trees and forests

## Table of Content


- Decision Trees
- Random Forests
- Boosted Decision Trees (XGBoost)
- Tutorial: Classifying images in the FMNIST dataset
- Feature Importance for Interpretable Machine Learning

## Intermezzo - Feature Engineering

A few comments on feature engineering.
- Belongs to the preprocessing part of the ML workflow. It is about making the ML task as easy as possible to train the model.
- It is about selecting the "best" features
- It is about transforming the features, for example, PCA, normalisation, scaling, one-hot encoding ...

**Normalisation** and **Scaling** may help the training not to focus more on features with big numbers than on features with smaller numbers.

For categorical data, the **one-hot encoding** serves a similar purpose.

Think of a two feature dataset that is distributed as two concentric regions with different regions. A transformation to spherical coordinates makes the classification trivial.  




## Decision Trees




With linear models one fits straight lines, or hyperplanes in higher dimensions, in the feature space. If the dataset boundaries don't follow straight lines, this is not optimal. Decision trees are models that overcome this limitation. They are

- fast to train
- easily interpretable
- non-linear and
- require small amount of data

They easily overfit. The remedy for that are forests. There are tree models for classification and regression. They can be trained (fitted) by examples (labeled/annotaded data) and thus belongs to the supervised learning paradigm.




The following example shows that each tree creates a partition of the feature space $X$ into subregions $R_i$, $i=1,...,N_R$, this partitioning being described by a tree structure.


Predictions will be made with the following procedure. Given a new test point:
1. Assign $x$ to the region it belongs, e.g. $R_k$
2. For classification, make a majority vote using the training points belonging to $R_k$. For regression, evaluate the mean of the target over all training points belonging to $R_k$.

Let's see how the trees generate a partition of the domain into distinct regions.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.datasets import make_blobs
from sklearn import tree

A function for helping us with the plotting:

In [ ]:
def plot_prediction_2d(x_min, x_max, y_min, y_max, classifier, ax=None):
  """
  Creates 2D mesh, predicts class for each point on the mesh, and visualises it
  """

  mesh_step = .02  # step size in the mesh
  x_coords = np.arange(x_min, x_max, mesh_step) # coordinates of mesh colums
  y_coords = np.arange(y_min, y_max, mesh_step) # coordinates of mesh rows

  # create mesh, and get x and y coordinates of each point point
  # arrenged as array of shape (n_mesh_rows, n_mesh_cols)
  mesh_nodes_x, mesh_nodes_y = np.meshgrid(x_coords, y_coords)

  # Plot the decision boundary. For that, we will assign a color to each
  # point in the mesh [x_min, x_max]x[y_min, y_max].

  # prepare xy pairs for prediction: matrix of size (n_mesh_rows*n_mesh_cols, 2)
  mesh_xy_coords = np.stack([mesh_nodes_x.flatten(),
                             mesh_nodes_y.flatten()], axis=-1)

  # obtain class for each node
  mesh_nodes_class = classifier.predict(mesh_xy_coords)


  # reshape to the shape (n_mesh_rows, n_mesh_cols)==mesh_nodes_x.shape for visualization
  mesh_nodes_class = mesh_nodes_class.reshape(mesh_nodes_x.shape)

  # Put the result into a color countour plot
  ax = ax or plt.gca()
  ax.contourf(mesh_nodes_x,
              mesh_nodes_y,
              mesh_nodes_class,
              cmap='Pastel1', alpha=0.5)

In [ ]:
# make 3-class dataset for classification

# Generate a synthetic dataset
centers = [[-5, 0], [0, 1.5], [5, -1]]
X, y = make_blobs(n_samples=1000, centers=centers, random_state=40)
transformation = [[0.4, 0.2], [-0.4, 1.2]]
X = np.dot(X, transformation)
dtcs = []
for depth in (1, 2, 3, 4):
    # do fit
    dtc = tree.DecisionTreeClassifier(max_depth=depth, criterion='gini', min_samples_leaf=3)
    dtcs.append(dtc)
    dtc.fit(X, y)

    # print the training scores (mean accuracy)
    print("training score : %.3f (depth=%d)" % (dtc.score(X, y), depth))

    # get range for visualization
    x_0 = X[:, 0]
    x_1 = X[:, 1]
    x_min = x_0.min() - 1
    x_max = x_0.max() + 1
    y_min = x_1.min() - 1
    y_max = x_1.max() + 1

    fig, ax = plt.subplots(1, 2,  figsize=(14,7), dpi=300)
    plot_prediction_2d(x_min, x_max, y_min, y_max, classifier=dtc, ax=ax[0])

    ax[0].set_title("Decision surface of DTC (%d)" % depth)

    # Plot also the training points
    colors = "rbg"
    for i, color in zip(dtc.classes_, colors):
        idx = np.where(y == i)
        ax[0].scatter(x_0[idx], x_1[idx], c=color,
                    edgecolor='black', s=20, linewidth=0.2)

    with plt.style.context('classic'):
      tree.plot_tree(dtc, ax=ax[1]);

    plt.tight_layout()
    plt.show()

We generated a synthetic dataset and used it to train various tree models. What we didn't do:
- split into training, validation and test sets
- check for overfitting
- compare with a baseline model (well, a tree with depth 1 may beconsidered as linear regression)

### How is the tree constructed from a labelled training set (we skip this, only for specially interested)?

#### Classification
Given a sample $S$ of $N$ points in feature space ($x_i$, $i=1...N$), each assigned to one of $C$ classes ($c_i \in \{1,..,C\}$), we compute class proportions:
$$p_c = \frac{N_c}{N}$$
where $N_c$ is the number of points in class $c$.

**Impurity measures:**
- **Gini index:**
  $$G(S) = 1 - \sum_c p_c^2$$
- **Entropy:**
  $$E(S) = -\sum_c p_c \log(p_c)$$

These quantify how mixed the classes are in a node.

**Splitting:**
- At each step, choose a feature and cutoff $a$.
- Points with $x_{i,k} < a$ go left; $x_{i,k} \ge a$ go right.
- Define $S_{<}$, $S_{>}$ and their sizes $N_{<}$, $N_{>}$.
- **Information gain:**
  $$ IG = E(S) - \frac{N_{<}}{N} E(S_{<}) - \frac{N_{>}}{N} E(S_{>}) $$
- (Same formula for Gini index.)
- The split with highest information gain is chosen, favoring relevant features at the top and helping reduce overfitting by pruning.

---
### Regression
Similar setup, but targets are real values $y_i \in \mathbb{R}$.

**Splitting criterion:**
- Use sample variance:
  $$V(S) = \frac{1}{N} \sum_i (y_i - \bar{y})^2$$
- After splitting, compute means $y_{<}$ and $y_{>}$ for left/right nodes.
- **Reduction in variance:**
  $$ \frac{N_{<}}{N} V(S_{<}) + \frac{N_{>}}{N} V(S_{>}) = \frac{1}{N}\left(\sum_{i \in S_{<}} (y_i-y_{<})^2 + \sum_{i \in S_{>}} (y_i-y_{>})^2\right) $$
- The best split maximizes reduction in variance.

---
These are known as the **CART** (Classification and Regression Trees) splitting criteria.

## Random Forests

The `sklearn.ensemble` provides several ensemble algorithms. RandomForest is an averaging algorithm based on randomized decision trees. This means a diverse set of classifiers is created by introducing randomness in the classifier construction.

The prediction of the ensemble is given as the averaged prediction of the individual classifiers (regression) or by majority voting (classification). E.g. for regression:

$$ RF(x) = \frac{1}{N_\text{trees}}\sum_{i=1}^{N_\text{trees}} Tree_i(x)$$

Individual decision trees typically exhibit high variance and tend to overfit.
In random forests:
* each tree in the ensemble is built from a sample drawn with replacement (i.e., a bootstrap sample) from the training set.
* when splitting each node during the construction of a tree, the best split is found from a random subset of features, according to `max_features` parameter.

The injected randomness in forests yield decision trees with somewhat decoupled prediction errors. By taking an average of those predictions, some errors can cancel out. Random forests achieve a reduced variance by combining diverse trees, sometimes at the cost of a slight increase in bias. In practice the variance reduction is often significant, hence yielding an overall better model.


In [ ]:
from sklearn import ensemble

In [ ]:
# make 3-class dataset for classification
centers = [[-5, 0], [0, 1.5], [5, -1]]
X, y = make_blobs(n_samples=1000, centers=centers, random_state=40)
transformation = [[0.4, 0.2], [-0.4, 1.2]]
X = np.dot(X, transformation)

for n_est in (1, 4, 50):
    # do fit
    rfc = ensemble.RandomForestClassifier(max_depth=4, n_estimators=n_est,)
    rfc.fit(X, y)

    # print the training scores
    print("training score : %.3f (n_est=%d)" % (rfc.score(X, y), n_est))

    # get range for visualization
    x_0 = X[:, 0]
    x_1 = X[:, 1]
    x_min = x_0.min() - 1
    x_max = x_0.max() + 1
    y_min = x_1.min() - 1
    y_max = x_1.max() + 1

    plt.figure(figsize=(5,5))
    plot_prediction_2d(x_min, x_max, y_min, y_max, classifier=rfc)

    # Plot also the training points
    colors = 'rbg'
    for i, color in enumerate(colors):
        idx = np.where(y == i)
        plt.scatter(x_0[idx], x_1[idx], c=color,
                    edgecolor='black', s=20, linewidth=0.2)


    plt.show()

In [ ]:
plt.figure(dpi=300)
with plt.style.context('classic'):
  tree.plot_tree(rfc.estimators_[20]);

## Boosted Decision Trees (XGBoost)

Another approach to the ensemble tree modeling is Boosted Decision Trees. In a boosting framework, the trees are created sequentially. This way each next tree reduces error of the ensemble, by adding corrections to previous predictions.

One of the most popular implementations of boosting is XGBoost:

https://arxiv.org/abs/1603.02754

https://xgboost.readthedocs.io/en/stable/python/python_api.html

### Classification on the Blobs Dataset

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.datasets import make_blobs

In [ ]:
from xgboost import XGBClassifier

# make 3-class dataset for classification
centers = [[-5, 0], [0, 1.5], [5, -1]]
X, y = make_blobs(n_samples=1000, centers=centers, random_state=40)
transformation = [[0.4, 0.2], [-0.4, 1.2]]
X = np.dot(X, transformation)

params = {
    'num_class': 3,               # Number of classes in the target variable
    'max_depth': 3,               # Maximum depth of trees
    'learning_rate': 0.1,         # Learning rate
    'n_estimators': 10
}
model = XGBClassifier(**params)
model.fit(X, y, verbose=True)

# print the training scores
print("training score : %.3f" % (model.score(X, y)))
x_0 = X[:, 0]
x_1 = X[:, 1]
x_min = x_0.min() - 1
x_max = x_0.max() + 1
y_min = x_1.min() - 1
y_max = x_1.max() + 1
plt.figure(figsize=(5,5))
plot_prediction_2d(x_min, x_max, y_min, y_max, classifier = model)

plt.title(f'Decision surface of Boosted model after 100 iterations')
plt.axis('tight')

# Plot also the training points
colors = 'rbg'
for i, color in enumerate(colors):
    idx = np.where(y == i)
    plt.scatter(x_0[idx], x_1[idx], c=color,
                edgecolor='black', s=20, linewidth=0.2)

# Plot the three one-against-all classifiers
xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()


plt.show()

### Regression on House Prices

#### Get dataset

Subset of the Ames Houses dataset: http://jse.amstat.org/v19n3/decock.pdf. First a method preparing the dataset:

In [ ]:
import pandas as pd

In [ ]:
def house_prices_dataset(return_df=False, return_df_xy=False, price_max=400000, area_max=40000, data_path='/content/data/data'):
  #path = os.path.join(data_path, 'AmesHousing.csv')
  path = "https://raw.githubusercontent.com/neworldemancer/DSF5/refs/heads/master/data/AmesHousing.csv"
  df = pd.read_csv(path, na_values=('NaN', ''), keep_default_na=False,  )
  rename_dict = {k:k.replace(' ', '').replace('/', '') for k in df.keys()}
  df.rename(columns=rename_dict, inplace=True)

  useful_fields = ['LotArea',
                  'Utilities', 'OverallQual', 'OverallCond',
                  'YearBuilt', 'YearRemodAdd', 'ExterQual', 'ExterCond',
                  'HeatingQC', 'CentralAir', 'Electrical',
                  '1stFlrSF', '2ndFlrSF','GrLivArea',
                  'FullBath', 'HalfBath',
                  'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
                  'Functional','PoolArea',
                  'YrSold', 'MoSold'
                  ]
  target_field = 'SalePrice'

  df.dropna(axis=0, subset=useful_fields+[target_field], inplace=True)

  cleanup_nums = {'Street':      {'Grvl': 0, 'Pave': 1},
                  'LotFrontage': {'NA':0},
                  'Alley':       {'NA':0, 'Grvl': 1, 'Pave': 2},
                  'LotShape':    {'IR3':0, 'IR2': 1, 'IR1': 2, 'Reg':3},
                  'Utilities':   {'ELO':0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3},
                  'LandSlope':   {'Sev':0, 'Mod': 1, 'Gtl': 3},
                  'ExterQual':   {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'ExterCond':   {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'BsmtQual':    {'NA':0, 'Po':1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex':5},
                  'BsmtCond':    {'NA':0, 'Po':1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex':5},
                  'BsmtExposure':{'NA':0, 'No':1, 'Mn': 2, 'Av': 3, 'Gd': 4},
                  'BsmtFinType1':{'NA':0, 'Unf':1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ':5, 'GLQ':6},
                  'BsmtFinType2':{'NA':0, 'Unf':1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ':5, 'GLQ':6},
                  'HeatingQC':   {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'CentralAir':  {'N':0, 'Y': 1},
                  'Electrical':  {'':0, 'NA':0, 'Mix':1, 'FuseP':2, 'FuseF': 3, 'FuseA': 4, 'SBrkr': 5},
                  'KitchenQual': {'Po':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'Functional':  {'Sal':0, 'Sev':1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2':5, 'Min1':6, 'Typ':7},
                  'FireplaceQu': {'NA':0, 'Po':1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex':5},
                  'PoolQC':      {'NA':0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex':4},
                  'Fence':       {'NA':0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv':4},
                  }

  df_X = df[useful_fields].copy()
  df_X.replace(cleanup_nums, inplace=True)  # convert continous categorial variables to numerical
  df_Y = df[target_field].copy()

  x = df_X.to_numpy().astype(np.float32)
  y = df_Y.to_numpy().astype(np.float32)

  if price_max>0:
    idxs = y<price_max
    x = x[idxs]
    y = y[idxs]

  if area_max>0:
    idxs = x[:,0]<area_max
    x = x[idxs]
    y = y[idxs]

  return (x, y, df) if return_df else ((x, y, (df_X, df_Y)) if return_df_xy else (x,y))

#### Fit and assess the model

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn import linear_model

x, y = house_prices_dataset()

# Split your data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

evals_result = {}
# Create an XGBoost regression model
#model = linear_model.LinearRegression()
#model = ensemble.RandomForestClassifier()
model = XGBRegressor(eval_metric='rmse' ,
                     max_depth = 5,
                     early_stopping_rounds = 10,
                     learning_rate = 0.3,
                     n_estimators = 100 )

# Train the XGBoost regression model
model.fit(x_train, y_train,
          eval_set=[(x_train, y_train),(x_test, y_test)],
          verbose=True)

y_p_train = model.predict(x_train)
y_p_test = model.predict(x_test)

In [ ]:
plt.plot(model.evals_result_['validation_0']['rmse'], label='train')
plt.plot(model.evals_result_['validation_1']['rmse'], label='test')
plt.legend()
plt.xlabel('Number of Estimators')
plt.ylabel('RMSE')
plt.show()

In [ ]:
from sklearn.metrics import r2_score

# mse
print('train mse =', np.std(y_train - y_p_train))
print('test mse =', np.std(y_test - y_p_test))

residuals=y_train - y_p_train
print('train mse =', np.sqrt( np.mean(residuals**2)) )
residuals=y_test - y_p_test
print('train mse =', np.sqrt( np.mean(residuals**2)) )

# mse
print('train mae =', np.mean(np.abs(y_train - y_p_train)))
print('test mae =', np.mean(np.abs(y_test - y_p_test)))
# R2
print('train R2 =', r2_score(y_train, y_p_train))
print('test R2 =', r2_score(y_test, y_p_test))


# 4. plot y vs predicted y for test and train parts
plt.figure(figsize=(5,5))
plt.plot(y_train, y_p_train, 'b.', label='train')
plt.plot(y_test, y_p_test, 'r.', label='test')

plt.plot(y_train, y_train,'-')

plt.plot([0], [0], 'w.')  # dummy to have origin
plt.xlabel('true')
plt.ylabel('predicted')
plt.gca().set_aspect('equal')
plt.legend()
plt.plot()

## Tutorial: Classifying images in the FMNIST dataset

In [ ]:
path_train = "https://media.githubusercontent.com/media/fpleoni/fashion_mnist/refs/heads/master/fashion-mnist_train.csv"
path_test  = "https://media.githubusercontent.com/media/fpleoni/fashion_mnist/refs/heads/master/fashion-mnist_test.csv"

fashion_train = pd.read_csv(path_train)
fashion_test = pd.read_csv(path_test)

print(fashion_train.shape)
print(fashion_test.shape)

target_train = fashion_train["label"]
features_train = fashion_train.drop(["label"], axis = 1)
target_test = fashion_test["label"]
features_test= fashion_test.drop(["label"], axis = 1)


In [ ]:
# View the first image
first_image = features_train.iloc[0]
first_image = np.array(first_image, dtype = "float")
pixels = first_image.reshape((28, 28))
plt.imshow(pixels, cmap = "gray_r")
plt.axis("off")
plt.show()
print ("Label:", target_train.iloc[0])

In [ ]:
import xgboost as xgb
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report, roc_auc_score
# Create XGB Classifier object
#xgb_clf = xgb.XGBClassifier(objective = "multi:softmax", tree_method = "gpu_exact",
#                            predictor = "gpu_predictor", verbosity = True)
xgb_clf = xgb.XGBClassifier()
# Fit model
xgb_model = xgb_clf.fit(features_train, target_train)
# Predictions
y_train_preds = xgb_model.predict(features_train)
y_test_preds = xgb_model.predict(features_test)

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
# Print F1 scores and Accuracy
print("Training F1 Micro Average: ", f1_score(target_train, y_train_preds, average = "micro"))
print("Test F1 Micro Average: ", f1_score(target_test, y_test_preds, average = "micro"))
print("Test Accuracy: ", accuracy_score(target_test, y_test_preds))

You may do the same with logistic regression to have a baseline to compare against.

## Exercise

Expand with own bullet points on the ML workflow considering a task on the FMNIST dataset.
- Task
- Data collection
- Preprocessing and explorative data analysis
- Training with different models
- Assessment table
- Interpretation (feature importance etc)